# STRATA: o `junta` novo contra o antigo, na mesma máquina do Colab

No 70B, voltar ao BF16 levou 6.128 s em 8 núcleos, lento demais para o argumento de download. O `junta` foi refeito:

- **leitura, decodificação e hash/gravação em fila**: um fio lê e confere o próximo tensor enquanto outro decodifica e um terceiro calcula os hashes e grava;
- **uma leitura só**: o SHA-256 dos arquivos inteiros (inclusive o GGUF) agora só roda com `--tudo`; cada byte que entra na saída continua conferido pelo hash do seu stream, e cada shard pelo SHA-256 dele;
- **decodificador do fecho ~27% mais rápido** (busca da oitava em O(1)), com saída idêntica.

No contêiner de 4 núcleos, o 8B foi de 1.353 s (antigo, `--tudo`, disco frio) para ~305 s. Este notebook mede as duas versões **no mesmo runtime do Colab** que rodou o 70B, com o Llama-3.1-8B (~25–35 min, ~35 GB de disco). A razão entre as duas dá a estimativa para o 70B sem gastar mais 5 h.

**Runtime:** o mesmo do 70B (CPU com RAM alta). No fim, a última célula baixa um zip; mande ele na thread.

In [ ]:
# 1. A máquina (o SHA-256 depende muito de o processador ter a extensão SHA)
import os, shutil, subprocess
def sh(cmd, mostra=True):
    r = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, executable="/bin/bash")
    if mostra:
        print(r.stdout[-4000:])
    if r.returncode:
        raise SystemExit(f"falhou: {cmd}")
    return r.stdout
PASTA = "/content/strata_ab"
os.makedirs(PASTA, exist_ok=True)
cpu = sh("lscpu | grep -E 'Model name|^CPU\\(s\\)|Thread'", False)
flags = open("/proc/cpuinfo").read()
info = f"{cpu.strip()}\nsha_ni: {'sha_ni' in flags} · avx2: {'avx2' in flags}\nRAM {os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9:.1f} GB · disco livre {shutil.disk_usage(PASTA).free / 1e9:.0f} GB"
print(info)

In [ ]:
# 2. Rust e as duas versões do strata-ref (antiga = commit do teste do 70B) (≈ 5 min)
RAMO, ANTIGO = "claude/project-thread-btiyii", "070e992"
if not os.path.exists(os.path.expanduser("~/.cargo/bin/cargo")):
    sh("curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
REPO, VELHO = "/content/strata", "/content/strata_antigo"
if not os.path.exists(REPO):
    sh(f"git clone -q --filter=blob:none -b {RAMO} https://github.com/cantyll/strata {REPO}")
if not os.path.exists(VELHO):
    sh(f"cd {REPO} && git worktree add -q {VELHO} {ANTIGO}")
sh(f"cd {REPO} && git log -1 --oneline && cd {VELHO} && git log -1 --oneline")
for r in (REPO, VELHO):
    sh(f"cd {r}/leitor-rs && cargo build --release -q 2>&1 | tail -3; ls -la target/release/strata-ref")
sh("pip install -q -U huggingface_hub hf_xet psutil", False)
NOVO_BIN, VELHO_BIN = f"{REPO}/leitor-rs/target/release/strata-ref", f"{VELHO}/leitor-rs/target/release/strata-ref"

In [ ]:
# 3. O teste completo no 8B com a versão nova: baixa, grava os estratos e volta (≈ 15 min)
import sys
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
cmd = [sys.executable, "-u", "tests/dist/teste_grande.py", "--hf", "unsloth/Meta-Llama-3.1-8B-Instruct",
       "--gguf-repo", "bartowski/Meta-Llama-3.1-8B-Instruct-GGUF", "--gguf", "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf",
       "--dir", PASTA, "--bin", NOVO_BIN]
p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for linha in p.stdout:
    print(linha, end="")
p.wait()
assert p.returncode == 0, "o teste falhou"

In [ ]:
# 4. As duas versões do junta sobre os mesmos arquivos (≈ 10–25 min)
import time, re
IDX = f"{PASTA}/dist/model.strata.index.json"
GG = f"{PASTA}/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"
def limpa_cache():
    subprocess.run("sync; echo 3 > /proc/sys/vm/drop_caches", shell=True, stderr=subprocess.DEVNULL)
def junta(bin_, extra, nome):
    limpa_cache()
    t = time.time()
    out = sh(f"{bin_} junta {IDX} --base {GG} {extra} 2>&1", False)
    s = time.time() - t
    open(f"{PASTA}/resultados/ab_{nome}.txt", "w").write(out)
    ok = len(re.findall(r"\(confere\)", out))
    print(f"{nome}: {s:.0f} s, {ok} shards conferidos\n  " + "\n  ".join(l for l in out.splitlines() if l.startswith(("ok:", "tempos:"))))
    return s
tempos = {
    "antigo --tudo (como no 70B)": junta(VELHO_BIN, "--tudo", "antigo_tudo"),
    "novo": junta(NOVO_BIN, "", "novo"),
    "novo --tudo": junta(NOVO_BIN, "--tudo", "novo_tudo"),
}

In [ ]:
# 5. Resumo, estimativa para o 70B e download
base = tempos["antigo --tudo (como no 70B)"]
linhas = [f"# junta: antigo × novo no Colab (Llama-3.1-8B)", "", "```", info, "```", "", "| versão | s | × |", "|---|---:|---:|"]
linhas += [f"| {k} | {v:.0f} | {base / v:.2f} |" for k, v in tempos.items()]
linhas += ["", f"70B, que levou 6.128 s com o antigo: estimativa com o novo ≈ {6128 * tempos['novo'] / base:.0f} s (proporção do 8B; o 70B não cabe no cache de disco, então pode ficar pior)."]
txt = "\n".join(linhas)
open(f"{PASTA}/resultados/ab.md", "w").write(txt + "\n")
from IPython.display import Markdown, display
display(Markdown(txt))
z = shutil.make_archive("/content/strata_junta_ab", "zip", f"{PASTA}/resultados")
try:
    from google.colab import files
    files.download(z)
except Exception:
    print(z)